In [1]:
# Calculate OSDMA8 population weighted exposure

In [2]:
import xarray as xr
import os
import numpy as np

In [3]:
# === Path config ===
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
MASK_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"
O3_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/"

In [4]:
pop_ssp2 = xr.open_dataarray(f"{POP_DIR}ssp2_total_regrid_annual_2000-2100.nc")
country_mask = xr.open_dataarray(f"{MASK_DIR}GBD_Country_Masks_0.10_popgrid_newlabels.nc")

In [5]:
# === Path config ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/ozone/exposure/"
# SCENARIOS = ["ARISE", "SSP245"]
SCENARIOS = ["SSP245"]


# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(2, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        # Load data array
        if scenario == "ARISE":
            dates = "2035-2068"
        elif scenario == "SSP245":
            dates = "2020-2068"

        o3 = xr.open_dataarray(f"{O3_DIR}OSDMA8_BC_popgrid_CESM2_{scenario}_{ens_num:02d}_{dates}.nc")
        population = pop_ssp2.sel(year=o3.year)  # Select same years as o3 data for population

        weighted_value = population * o3

        country_list = []

        for i in range(204):
            mask = country_mask.isel(country=i)
            country_weight = xr.where(mask == 1, weighted_value, np.nan).sum(dim=("lat", "lon"))
            pop_country = xr.where(mask == 1, population, np.nan).sum(dim=("lat", "lon"))
            country_pop_weighted = country_weight / pop_country
            country_list.append(country_pop_weighted)

        pop_weighted_exposure = xr.concat(country_list, "country")

        out_file = f"OSDMA8_country_population_weighted_exposure_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving to {out_path}")
        pop_weighted_exposure.to_netcdf(out_path)

Processing SSP245, Ensemble 02
Saving to /glade/work/awells/air_quality/CESM/ozone/exposure/OSDMA8_country_population_weighted_exposure_CESM2_SSP245_02_2020-2068.nc
Processing SSP245, Ensemble 03
Saving to /glade/work/awells/air_quality/CESM/ozone/exposure/OSDMA8_country_population_weighted_exposure_CESM2_SSP245_03_2020-2068.nc
Processing SSP245, Ensemble 04
Saving to /glade/work/awells/air_quality/CESM/ozone/exposure/OSDMA8_country_population_weighted_exposure_CESM2_SSP245_04_2020-2068.nc
Processing SSP245, Ensemble 05
Saving to /glade/work/awells/air_quality/CESM/ozone/exposure/OSDMA8_country_population_weighted_exposure_CESM2_SSP245_05_2020-2068.nc
Processing SSP245, Ensemble 06
Saving to /glade/work/awells/air_quality/CESM/ozone/exposure/OSDMA8_country_population_weighted_exposure_CESM2_SSP245_06_2020-2068.nc
Processing SSP245, Ensemble 07
Saving to /glade/work/awells/air_quality/CESM/ozone/exposure/OSDMA8_country_population_weighted_exposure_CESM2_SSP245_07_2020-2068.nc
Processing